# Серафим 1.5B → GGUF конвертация
**Что делает:** загружает обученный LoRA-адаптер, склеивает с базовой моделью, сохраняет в GGUF для Orange Pi 5.
**Время:** ~15 минут на T4 GPU.

In [ ]:
# 1. Установка
!pip install -q unsloth


In [ ]:
# 2. Загружаем адаптер с GitHub
import subprocess, os
if not os.path.exists('adapter_model.safetensors'):
    # Скачиваем zip адаптера
    url = 'https://github.com/unidel2035/gift/raw/main/data/lora/serafim-1.5b/gift-serafim-1.5b-lora.zip'
    subprocess.run(['wget', '-q', url, '-O', 'adapter.zip'])
    import zipfile
    with zipfile.ZipFile('adapter.zip') as z: z.extractall('.')
    print('Адаптер загружен')
else:
    print('Адаптер уже на месте')


In [ ]:
# 3. ГРУЗИМ МОДЕЛЬ + АДАПТЕР + КОНВЕРТИРУЕМ В GGUF
from unsloth import FastLanguageModel
import torch

print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Загружаем Qwen2.5-1.5B + LoRA адаптер...')

# Загружаем БАЗОВУЮ модель
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
    max_seq_length=512, dtype=None, load_in_4bit=True,
)

# Загружаем LoRA адаптер поверх
model.load_adapter('.')
FastLanguageModel.for_inference(model)
print('Модель + адаптер загружены')

# Конвертируем в GGUF
print('Конвертируем в GGUF Q4_K_M...')
model.save_pretrained_gguf(
    './serafim-gguf',
    tokenizer,
    quantization_method='q4_k_m',
)

# Показываем результат
import os
for f in os.listdir('./serafim-gguf'):
    size = os.path.getsize(f'./serafim-gguf/{f}') / 1e6
    print(f'  {f}: {size:.0f} MB')
print('\nГОТОВО! Скачай файл .gguf → залей на Orange Pi 5')
